# Working Memory Report — MazeHard-Reasoning

**Evaluating slow/fast latent dynamics, halting behavior, and solution-path
prediction in MazeHard as evidence for PFC-like structured working memory.
Stronger slot-addressability tests (cue-recall, item-position binding) are
complementary future work.**

> **Report type:** Internal scientific evidence audit.
> **Scientific question:** Does HRM exhibit the latent dynamics expected from
> a PFC-like hierarchical working-memory engine: slow H-state for context/plan
> memory, fast L-state for local computation, and adaptive halting during
> structured problem solving?
> **Current scope:** Behavioral prediction (MazeHard solution overlays) +
> H/L latent dynamics + halting diagnostics.
> **Not yet covered:** H/L decoding, flat RNN baseline, cue-recall slot
> tests, causal perturbations. See §11 for gap analysis.

This report defines the evidence contract for evaluating HRM as a PFC-like
hierarchical working-memory and reasoning substrate. Whittington Tale defines
the representational target: structured activity state. MazeHard supports
indirect tests through zH/zL persistence, H/L dissociation, prediction
behavior, and adaptive computation. In specification mode (no HRM ReportRun
present), sections display their expected contracts rather than data.


## 1. Report Identity and Execution Status

| Field              | Value                                               |
| ------------------ | --------------------------------------------------- |
| Model family       | HRM v1 (primary), HRM v2 (comparison, if available) |
| Checkpoint         | `checkpoints/hrm-v1/eval-weights-only.pt`           |
| Task               | MazeHard (solution-path prediction)                 |
| Split              | `test`                                              |
| Eval slice         | 4 diagnostic cases (`mazehard_n4`)                  |
| ReportRun root     | `artifacts/reports/hrm_v1/mazehard_n4`              |
| Eval artifact root | `artifacts/evaluation/hrm_v1/mazehard_n4`           |
| Created            | TBD                                                 |
| Device / precision | CPU (diagnostic regime)                             |
| Seed               | Not recorded in provenance                          |


## 2. Scientific Question

> Does the HRM model provide a plausible PFC-like hierarchical reasoning engine
> for structured tasks?

### Subquestions

| ID  | Question                                                       | Evidence section |
| --- | -------------------------------------------------------------- | ---------------- |
| Q1  | Does HRM solve MazeHard above baseline?                        | §6               |
| Q2  | Are predicted solution overlays coherent and path-like?        | §7               |
| Q3  | Does `z_H` show slow, persistent working-memory dynamics?      | §8               |
| Q4  | Does `z_L` show fast, transient computation dynamics?          | §9               |
| Q5  | Does the model use adaptive halting in a task-sensitive way?   | §10              |
| Q6  | What remains untested about Whittington-style activity slots?  | §11              |

**Scope warning:** This notebook evaluates HRM as a PFC-like hierarchical
working-memory and reasoning substrate. Whittington Tale defines the
representational target: structured activity state. MazeHard supports
indirect tests through zH/zL persistence, H/L dissociation, prediction
behavior, and adaptive computation. Stronger slot-addressability tests —
cue recall, item-position binding, counterfactual cueing, and causal
perturbation — require future infrastructure.


### 2.1 Paper Context: Wang HRM + Whittington Tale

**Wang et al. 2025 (HRM)** answers an implementation question:
How can a model perform latent recurrent reasoning efficiently?
Answer: hierarchical slow/fast recurrent computation with adaptive halting.

**Whittington et al. 2025 (Tale)** answers a representation question:
What should PFC working memory contain?
Answer: structured, controllable activity-based variables (activity slots).

**This notebook** evaluates whether HRM latent recurrent states provide
evidence for PFC-like structured working-memory dynamics during structured
reasoning. Whittington Tale defines the representational target: structured
activity state. MazeHard supports indirect tests through zH/zL persistence,
H/L dissociation, prediction behavior, and adaptive computation.

### Role Summary

| Paper                   | Role in this notebook                                      |
| ----------------------- | ---------------------------------------------------------- |
| Wang et al. 2025        | Primary: model architecture and dynamics                   |
| Whittington et al. 2025 | Primary: representational target for working-memory diagnostics |


### 2.3 Evidence Ladder / Evidence Standard

| Level | Name                             | Description                                               | Target notebook |
| ----- | -------------------------------- | --------------------------------------------------------- | --------------- |
| 0     | Behavioral competence            | HRM solves MazeHard above baseline                        | 02 (this)       |
| 1     | Observable hierarchical dynamics | `z_H` and `z_L` show different temporal profiles          | 02 (this)       |
| 2     | Adaptive control                 | Halting varies with task conditions                       | 02 (this)       |
| 3     | Functional interpretation        | `z_H` appears slow/stable; `z_L` appears fast/transient   | 02 (this)       |
| 4     | Decodable task variables         | Goal, path state, next action decodable from `z_H`/`z_L`  | 02 (future)     |
| 5     | Structured PFC slots             | Item/role/sequence in addressable, controllable subspaces | 03 (future)     |
| 6     | Causal necessity                 | Perturbing slot degrades behavior predictably             | 03 (future)     |

**This notebook targets Levels 0–3.** Level 4 requires decoder infrastructure
not yet built. Levels 5–6 require WM-specific tasks and causal interventions
planned for complementary future infrastructure.


## 3. Task and Benchmark — MazeHard

MazeHard is a structured pathfinding task requiring multi-step reasoning
over a maze-like symbolic environment. The model receives the maze layout
as input and must predict the solution path token-by-token.

### Why MazeHard for PFC/HRM

- Tests hierarchical recurrent computation (Wang HRM).
- Requires multi-step state maintenance and local action selection.
- Provides `z_H`, `z_L`, and halting traces for dynamics analysis.

### Limitations for isolated slot-addressability tests

MazeHard supports indirect working-memory evidence (state persistence,
H/L dissociation, adaptive control) but does not isolate slot-addressable
memory from reasoning. Stronger tests require:
- Cue-dependent recall (isolates retrieval from encoding)
- Item-position binding (tests structured representational slots)
- Counterfactual cue interventions (tests controllability)
- Delay-period distractors (tests active maintenance)

### Benchmark Parameters

| Parameter           | Value                              |
| ------------------- | ---------------------------------- |
| Task family         | `mazehard`                         |
| Split               | `test`                             |
| Cases in this slice | 4 (diagnostic, not full benchmark) |
| Provider            | `MazeHardReplayProvider`           |

> **Note:** Full task metadata (maze sizes, path length distribution,
> branching factor) should be populated from the eval artifact when available.


## 4. Model Internals Inspected

The HRM architecture separates high-level context/plan states (`z_H`) from
low-level computation states (`z_L`) via a hierarchical recurrent design.
This report inspects the following internal variables and pathways:

| System       | Scientific role                    | Trace key                | Report-facing name                | Evidence section |
| ------------ | ---------------------------------- | ------------------------ | -------------------------------- | ---------------- |
| H-state      | Slow context / plan / WM state     | `pfc/z_H`                | High-level recurrent state        | §8               |
| L-state      | Fast local computation / update    | `pfc/z_L`                | Low-level recurrent state         | §9               |
| Halting      | Adaptive computation / termination | `act/halted`             | Halting timeline                  | §10              |
| Prediction   | Behavioral solution hypothesis     | `pred/solution_overlay`  | MazeHard solution overlay         | §7               |

### 4.1 Theory-to-Measurement Mapping

| Concept                           | Trace / Metric                  | Evidence status | Section |
| --------------------------------- | ------------------------------- | --------------- | ------- |
| Slow PFC controller state         | `pfc/z_H` norm and delta        | testable now    | §8      |
| Fast local computation            | `pfc/z_L` norm and delta        | testable now    | §9      |
| Hierarchical timescale separation | `h_l_delta_ratio`               | testable now    | §8–9    |
| Adaptive computation / stopping   | `act/halted`                    | testable now    | §10     |
| Behavioral solution quality       | `pred/solution_overlay`         | testable now    | §6–7    |
| Path prediction dynamics          | `mazehard_prediction_evolution` | testable now    | §7      |
| Plan/content decodability         | decoder over `z_H` / `z_L`      | future infra    | §11     |
| Structured activity slots         | cue-recall / binding task       | complementary future work | §11 |
| Causal slot necessity             | perturbation / ablation         | future work     | §11     |

### Trace Shape Reference

| Trace key               | Shape          | Description                     |
| ----------------------- | -------------- | ------------------------------- |
| `pfc/z_H`               | `(T, B, S, D)` | High-level recurrent state      |
| `pfc/z_L`               | `(T, B, S, D)` | Low-level recurrent state       |
| `act/halted`            | `(T, B, S)`    | Binary halting signal           |
| `pred/solution_overlay` | `(T, B, S)`    | Predicted solution-path overlay |


### 4.4 Artifact and Figure Contract

### Available Figures (when report exists)

| Figure ID                       | Maturity     | Description                            |
| ------------------------------- | ------------ | -------------------------------------- |
| `mazehard_solution_overlay`     | stable       | GT vs predicted solution-path overlays |
| `mazehard_prediction_evolution` | experimental | Per-step prediction evolution          |
| `pfc_latent_dynamics`           | stable       | H/L norm and delta over rollout time   |
| `halting_timeline`              | stable       | Binary halting signal heatmap          |
| `halt_logit_evolution`          | experimental | Halt/continue logits over steps        |

### Available Diagnostics

| Metric               | Description                  |
| -------------------- | ---------------------------- |
| `h_state_norm_mean`  | Mean L2 norm of `z_H`        |
| `l_state_norm_mean`  | Mean L2 norm of `z_L`        |
| `h_state_delta_mean` | Mean temporal delta of `z_H` |
| `l_state_delta_mean` | Mean temporal delta of `z_L` |
| `h_l_delta_ratio`    | Ratio of H delta to L delta  |

### Not Yet Available

| Capability                  | Status      |
| --------------------------- | ----------- |
| H/L linear decoding         | no infra    |
| Subspace / binding analysis | no infra    |
| Flat RNN/GRU baseline       | no eval     |
| Cue-recall eval pipeline    | complementary future infra |
| Causal perturbation         | future work |


## 5. Training Dynamics and Checkpoint Selection

| Field                     | Status                                                                                                                                                                                |
| ------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| ReportRun-backed          | ❌ Pending                                                                                                                                                                            |
| Source needed             | Training logs / TensorBoard / Lightning metrics                                                                                                                                       |
| Current blocker           | Training logs are not yet ingested into the ReportRun artifact                                                                                                                        |
| Interpretation constraint | Behavioral and representational results below describe the selected checkpoint only. They do **not** establish convergence, and the reader cannot distinguish learned structure from overfit memorisation on this small eval slice without training curves. |

**Required diagnostics (future):**

- `train/loss`, `val/loss`
- Prediction accuracy over training steps
- Loss components (token prediction, halting, auxiliary)
- Learning rate schedule
- Gradient norm history

> Until training curves are available, the checkpoint selection is
> unvalidated. Results in the sections below should be read as describing a
> single checkpoint, not a converged model.


## 6. Behavioral Performance

The primary metric for MazeHard is `sequence_exact` — whether the predicted
solution path exactly matches the ground-truth path. Secondary metrics include
`token_accuracy` and `success_rate`.

**Chance baseline:** Depends on vocabulary size and maze difficulty. For a
token-level uniform baseline the expected accuracy is low.

> Behavioral success is a prerequisite for meaningful latent-dynamics analysis.
> If HRM does not solve MazeHard above baseline, the notebook should primarily
> serve as a failure analysis rather than claiming PFC-like dynamics.


In [2]:
if primary_report is not None:
    metrics = primary_report.metrics()
    print("### Primary Metrics")
    print()
    for m in metrics:
        hb = " (higher is better)" if m.higher_is_better else ""
        print(f"  {m.metric:<30s} {m.value:<8} {m.unit or '---':<10s} {hb}")
else:
    print("> Specification mode -- metrics not available.")
    print("> Expected metrics: `sequence_exact`, `token_accuracy`, `success_rate`")

> Specification mode -- metrics not available.
> Expected metrics: `sequence_exact`, `token_accuracy`, `success_rate`


## 7. Prediction Examples and Error Taxonomy

This section shows behavioral prediction overlays — per-step comparisons
between ground-truth solution paths and model-predicted paths.

### Error Taxonomy

| Error category          | Observable signature                         | PFC/HRM interpretation                        |
| ----------------------- | -------------------------------------------- | --------------------------------------------- |
| Correct solution        | Predicted path matches GT                    | Successful hierarchical computation           |
| Near-miss               | Mostly correct, minor deviations             | Plan OK, local execution imperfect            |
| Globally wrong          | Path goes to wrong region                    | Possible high-level plan failure (`z_H`)      |
| Locally greedy          | Correct direction, wrong final path          | Possible low-level refinement failure (`z_L`) |
| Looping / perseveration | Repeated path segments                       | Unstable state or poor halting control        |
| Empty / no prediction   | Model halted immediately or produced nothing | Halting or initialization failure             |


In [3]:
if primary_report is not None:
    from IPython.display import Image, display

    entry = primary_report.figure_entry(
        "mazehard_solution_overlay", preferred_format="png"
    )
    if entry:
        abs_path = primary_report.root / entry.path
        print(f"mazehard_solution_overlay (case={entry.regime_id})")
        display(Image(filename=str(abs_path)))
    else:
        print("mazehard_solution_overlay not available.")
else:
    print("> Specification mode -- figures not available.")
    print("> Expected figure: `mazehard_solution_overlay`")

> Specification mode -- figures not available.
> Expected figure: `mazehard_solution_overlay`


## 8. H-state Diagnostics — Slow Working-Memory / Plan State

This section evaluates whether the HRM high-level state (`z_H`) behaves like
a slow, persistent PFC working-memory state. In the Wang HRM framing, `z_H`
should change more slowly than `z_L` and preserve task-level context across
recurrent computation. In the Whittington framing, this is the candidate
substrate in which structured working-memory variables could be maintained.

**Expected signature:**
- `z_H` has lower temporal delta than `z_L`.
- `z_H` remains stable across local recurrent updates.
- `z_H` dynamics are compatible with persistent task context or plan state.

**What this shows:**
- Whether the high-level recurrent state has slow temporal dynamics.
- Whether the checkpoint exposes measurable H-state trajectories.

**What this does not show:**
- Whether `z_H` semantically encodes goal, plan, item, or slot variables.
- Whether `z_H` is causally necessary for behavior.
- Whether `z_H` contains addressable activity slots.

> **Caveat:** Norm and delta dynamics show temporal structure, not semantic
> content. They cannot prove planning or working memory by themselves.
> This is Level 1–2 evidence under §2.3.


## 9. L-state Diagnostics — Fast Computation / Local Update State

This section evaluates whether the HRM low-level state (`z_L`) behaves like
a fast, transient computation state. In the Wang HRM framing, `z_L` should
support local refinement within the slower context set by `z_H`. In the
Whittington framing, this is the candidate substrate for local updates,
manipulation, or action-relevant transformation of active working-memory
contents.

**Expected signature:**
- `z_L` changes faster than `z_H`.
- `z_L` shows stronger within-cycle fluctuation.
- `z_L` dynamics are compatible with local computation or execution-level
  refinement.

**What this shows:**
- Whether the low-level recurrent state has fast temporal dynamics.
- Whether `z_L` differs empirically from `z_H`.

**What this does not show:**
- Whether `z_L` encodes next action, local position, or retrieved content.
- Whether `z_L` implements slot updates.
- Whether the H/L division is functionally necessary.

> **Caveat:** Norm and delta dynamics show temporal structure, not semantic
> content. They cannot prove local execution or slot manipulation by
> themselves. This is Level 1–2 evidence under §2.3.


In [4]:
if primary_report is not None:
    from IPython.display import Image, display

    entry = primary_report.figure_entry("pfc_latent_dynamics", preferred_format="png")
    if entry:
        abs_path = primary_report.root / entry.path
        print(f"pfc_latent_dynamics (case={entry.regime_id})")
        display(Image(filename=str(abs_path)))

    # Also show scalar dynamics metrics
    metrics = primary_report.metrics()
    dynamics_metrics = [
        m
        for m in metrics
        if m.metric
        in (
            "h_state_norm_mean",
            "l_state_norm_mean",
            "h_state_delta_mean",
            "l_state_delta_mean",
            "h_l_delta_ratio",
        )
    ]
    if dynamics_metrics:
        print()
        print("### H/L Dynamics Summary Metrics")
        print()
        for m in dynamics_metrics:
            print(f"  {m.metric:<25s} {m.value:<10.4f} {m.unit or 'a.u.'}")
else:
    print("> Specification mode -- pfc_latent_dynamics not available.")
    print("> Expected figure: `pfc_latent_dynamics` (3-panel: norm, delta, summary)")

> Specification mode -- pfc_latent_dynamics not available.
> Expected figure: `pfc_latent_dynamics` (3-panel: norm, delta, summary)


## 10. Halting and Adaptive Computation

The `act/halted` trace records when the model decides to stop recurrent
computation. Adaptive halting is a form of metacognitive control — the
model decides when it has reasoned enough. This is consistent with PFC's
role in monitoring and regulating cognitive processes (Shenhav et al. 2013,
Expected Value of Control).

### Questions this section should answer (when data available)

1. Does the model halt at all, or does it run to the maximum step limit?
2. Does halting vary across cases (suggesting task-sensitive control)?
3. Does harder input (longer path, more obstacles) require more steps?
4. Is premature halting associated with incorrect solutions?


In [ ]:
if primary_report is not None:
    from IPython.display import Image, display

    entry = primary_report.figure_entry("halting_timeline", preferred_format="png")
    if entry:
        abs_path = primary_report.root / entry.path
        print(f"halting_timeline (case={entry.regime_id})")
        display(Image(filename=str(abs_path)))
    else:
        print("halting_timeline not available.")
else:
    print("> Specification mode -- halting_timeline not available.")
    print("> Expected figure: `halting_timeline`")

> Specification mode -- halting_timeline not available.
> Expected figure: `halting_timeline`


## 11. Evidence Gaps and Next Steps

| Missing claim                      | Why it matters                                        | Current blocker                     | Next artifact needed                   |
| ---------------------------------- | ----------------------------------------------------- | ----------------------------------- | -------------------------------------- |
| `z_H` encodes plan-level variables | Needed for PFC controller interpretation              | No decoder infrastructure           | `diagnostics/wm_decoding.py`           |
| `z_L` encodes local execution      | Needed for H/L functional dissociation                | No decoder infrastructure           | `diagnostics/wm_decoding.py`           |
| HRM-specific dynamics              | Need to rule out generic recurrent dynamics           | No flat RNN/GRU baseline eval       | Flat RNN trained checkpoint + eval run |
| Whittington-style activity slots   | Core PFC representation claim from Tale 2025          | No cue-recall eval task in pipeline | Cue-recall task provider + eval config |
| Cue-controllable slot access       | Needed for \"controllable\" part of Tale claim          | No counterfactual cue test infra    | Complementary future infra                            |
| Causal necessity of `z_H`/`z_L`    | Needed for mechanistic (not just correlational) claim | No perturbation/ablation infra      | Ablation config + eval run             |

### Priority Order for Next Implementation

1. **Generate HRM MazeHard eval artifacts** — unblocks all display sections.
2. **Build H/L decoder infrastructure** — enables Level 4 evidence.
3. **Train and evaluate flat RNN/GRU baseline** — enables specificity claim.
4. **Implement cue-recall eval pipeline** — enables complementary future WM tasks.
5. **Build perturbation/ablation infrastructure** — enables Level 6 evidence.


## 12. Evidence Summary

| Claim                                        | Variable / Pathway                    | Current status              | Verdict                                                          |
| -------------------------------------------- | ------------------------------------- | --------------------------- | ---------------------------------------------------------------- |
| HRM MazeHard behavior is measurable          | Output predictions                    | No HRM ReportRun present    | ⚠️ Not yet evaluated in current artifact state                  |
| Solution prediction overlays are inspectable | `pred/solution_overlay`               | No HRM ReportRun present    | ⚠️ Not yet evaluated in current artifact state                  |
| `z_H` and `z_L` traces are collectible       | `pfc/z_H`, `pfc/z_L`                  | No HRM ReportRun present    | ⚠️ Not yet evaluated in current artifact state                  |
| H/L temporal dynamics are measurable         | `pfc/z_H`, `pfc/z_L`, `h_l_delta_ratio`| No HRM ReportRun present   | ⚠️ Not yet evaluated in current artifact state                  |
| Adaptive halting is measurable               | `act/halted`                           | No HRM ReportRun present    | ⚠️ Not yet evaluated in current artifact state                  |
| Wang-style hierarchical timescale separation | `pfc/z_H`, `pfc/z_L`, `h_l_delta_ratio`| No HRM ReportRun present   | ⚠️ Not yet evaluated in current artifact state                  |
| PFC-like reasoning-engine interpretation     | H/L dynamics + halting                 | compatible, not proven      | ⚠️ Suggestive / conditional — compatible but not causally tested |
| `z_H` encodes plan-level variables           | `pfc/z_H`                              | No decoder infrastructure   | ❌ Not yet evaluated                                              |
| Whittington-style activity slots             | zH/zL dynamics + halting               | Working-memory signatures testable via MazeHard; full slot-addressability requires cue-recall tasks | ⚠️ Partial — working-memory dynamics testable; isolated slot-control requires future infrastructure |
| Causal role of `z_H`/`z_L` is established    | Perturbation / ablation                | No perturbation infra       | ❌ Not yet evaluated                                              |

**Bottom line:** This notebook currently defines the evidence contract for evaluating HRM as a PFC-like hierarchical working-memory and reasoning substrate. Whittington Tale defines the representational target: structured activity state. Existing code supports zH/zL dynamics, halting, and MazeHard solution overlays, but no HRM ReportRun is present yet; therefore behavioral and latent-dynamics claims remain unevaluated until the MazeHard artifacts are generated.

## 13. Reproducibility

### Artifact Paths

| Artifact                | Path                                       |
| ----------------------- | ------------------------------------------ |
| HRM v1 ReportRun        | `artifacts/reports/hrm_v1/mazehard_n4`     |
| HRM v1 eval artifacts   | `artifacts/evaluation/hrm_v1/mazehard_n4`  |
| HRM v1 checkpoint       | `checkpoints/hrm-v1/eval-weights-only.pt`  |
| HRM v1 model config     | `config/models/hrm-v1-base.toml`           |
| HRM v1 report spec      | `config/reporting/hrm_v1_mazehard_n4.toml` |
| MazeHard benchmark spec | `spec/spec-benchmark-suite.md`             |

### Commands to Reproduce

```bash
# Generate evaluation artifacts with all diagnostic traces (n=4)
python scripts/eval/run_eval.py \
    --model-family hrm-v1 \
    --checkpoint checkpoints/hrm-v1/eval-weights-only.pt \
    --config config/eval/hrm-v1-mazehard.toml \
    --task mazehard \
    --provider-ref ehc_sn.tasks.mazehard.providers.MazeHardReplayProvider \
    --regime-id mazehard_n4 --regime-kind diagnostic \
    --output artifacts/evaluation/hrm_v1/mazehard_n4 \
    --device cpu

# Generate ReportRun from eval artifacts
python scripts/reporting/run_report.py \
    --config config/reporting/hrm_v1_mazehard_n4.toml
```

### Known Reproducibility Gaps

| Gap                            | Impact                                                          |
| ------------------------------ | --------------------------------------------------------------- |
| Git commit SHA not recorded    | Cannot pin the exact code version                               |
| Checkpoint SHA256 not recorded | Cannot verify checkpoint integrity                              |
| Random seed not recorded       | Cannot reproduce stochastic elements                            |

| HRM v2 / EHC v1 checkpoints    | Not yet evaluated; hierarchical-reasoning comparison incomplete |